# Clase 3 · Tarea 02 — Mini proyecto: procesador de pedidos

### Ciclos, funciones y lambdas integradas

En esta tarea construirás **pieza por pieza** un procesador de pedidos
que toma una lista de cadenas crudas y produce una factura formateada.

```
  raw = ["laptop:2500000:2", "mouse:45000:5", "cargador:80000:0"]
    │
    │ parsear_pedidos
    ▼
  [{"producto": "laptop", "precio": 2500000, "cantidad": 2}, ...]
    │
    │ filtrar_nulos
    ▼
  [pedidos con cantidad > 0]
    │
    │ map(aplicar_descuento_volumen)
    ▼
  [pedidos + precio_con_descuento + descuento_pct]
    │
    │ calcular_subtotales
    ▼
  [pedidos + subtotal]
    │
    │ resumen_factura
    ▼
  {lineas, subtotal_neto, iva, total, ahorro_total}
    │
    │ imprimir_factura (todo el pipeline)
    ▼
  FACTURA impresa en pantalla
```

**Instrucciones**

1. Implementa cada función en orden (cada parte usa las anteriores).
2. **No cambies los nombres** de las funciones ni de los campos de los dicts.
3. Ejecuta los tests de cada parte hasta ver ✅.
4. La Parte 6 integra todo: pasa si y solo si las partes 1-5 son correctas.

> 🧠 Antes de programar cada función, lee su enunciado con atención al
> contrato: qué recibe, qué devuelve, qué campos añade.

In [ ]:
# Identifícate (esto ayuda si entregas el notebook).
NOMBRE = ""   # ✏️ escribe tu nombre completo
print("Tarea lista para resolver. Ejecuta cada bloque de tests tras implementar.")

## Ejercicio 1 · Parsear la lista de pedidos

Los pedidos llegan como una lista de cadenas con el formato `"producto:precio_unitario:cantidad"`. Por ejemplo:
`["laptop:2500000:2", "mouse:45000:5", "cargador:80000:0"]`

Implementa `parsear_pedidos(raw)` que devuelva una **lista de diccionarios** con claves `producto` (str), `precio` (int) y `cantidad` (int).

**Ejemplo:**
`parsear_pedidos(["laptop:2500000:2"])` → `[{'producto': 'laptop', 'precio': 2500000, 'cantidad': 2}]`

In [ ]:
def parsear_pedidos(raw):
    resultado = []
    for linea in raw:
        partes = linea.split(':')
        resultado.append({
            'producto': partes[0],
            'precio':   int(partes[1]),
            'cantidad': int(partes[2]),
        })
    return resultado

In [ ]:
# === Tests visibles · Ejercicio 1 ===
_p = parsear_pedidos(['laptop:2500000:2'])
assert _p == [{'producto': 'laptop', 'precio': 2500000, 'cantidad': 2}]
print("✅ Ejercicio 1: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 1 ===
_p2 = parsear_pedidos(['mouse:45000:5', 'cargador:80000:0'])
assert _p2[0]['producto'] == 'mouse'
assert _p2[1]['cantidad'] == 0
assert parsear_pedidos([]) == []
print("✅ Ejercicio 1: tests adicionales superados.")

## Ejercicio 2 · Filtrar pedidos nulos

Un pedido 'nulo' es aquel con `cantidad == 0`. Implementa `filtrar_nulos(pedidos)` que devuelva una **nueva lista** sin los pedidos nulos, usando `filter` y `lambda`.

**Ejemplo:** dado `[{'producto': 'a', 'precio': 100, 'cantidad': 2}, {'producto': 'b', 'precio': 50, 'cantidad': 0}]`, devuelve solo el primer elemento.

In [ ]:
def filtrar_nulos(pedidos):
    return list(filter(lambda p: p['cantidad'] > 0, pedidos))

In [ ]:
# === Tests visibles · Ejercicio 2 ===
_raw = ['laptop:2500000:2', 'mouse:45000:5', 'cargador:80000:0']
_pedidos = parsear_pedidos(_raw)
_activos = filtrar_nulos(_pedidos)
assert len(_activos) == 2
assert all(p['cantidad'] > 0 for p in _activos)
print("✅ Ejercicio 2: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 2 ===
assert filtrar_nulos([]) == []
_todos_nulos = [{'producto': 'x', 'precio': 10, 'cantidad': 0}]
assert filtrar_nulos(_todos_nulos) == []
print("✅ Ejercicio 2: tests adicionales superados.")

## Ejercicio 3 · Aplicar descuento por volumen

Los descuentos por volumen se aplican sobre el **precio unitario**:
- cantidad >= 10 → 15% de descuento
- cantidad >= 5  → 10% de descuento
- cantidad < 5   → sin descuento

Implementa `aplicar_descuento_volumen(pedido)` que reciba un **diccionario de pedido** y devuelva un **nuevo diccionario** con los mismos campos más `'precio_con_descuento'` (int) y `'descuento_pct'` (float, ej: 0.10).

**Ejemplo:** `{'producto': 'mouse', 'precio': 45000, 'cantidad': 5}` → añade `precio_con_descuento: 40500, descuento_pct: 0.10`.

In [ ]:
def aplicar_descuento_volumen(pedido):
    cantidad = pedido['cantidad']
    if cantidad >= 10:
        tasa = 0.15
    elif cantidad >= 5:
        tasa = 0.10
    else:
        tasa = 0.0
    precio_desc = int(pedido['precio'] * (1 - tasa))
    return {**pedido, 'precio_con_descuento': precio_desc, 'descuento_pct': tasa}

In [ ]:
# === Tests visibles · Ejercicio 3 ===
_m = aplicar_descuento_volumen({'producto': 'mouse', 'precio': 45000, 'cantidad': 5})
assert _m['precio_con_descuento'] == 40500
assert _m['descuento_pct'] == 0.10
print("✅ Ejercicio 3: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 3 ===
_l = aplicar_descuento_volumen({'producto': 'laptop', 'precio': 2500000, 'cantidad': 10})
assert _l['precio_con_descuento'] == 2125000
assert _l['descuento_pct'] == 0.15
_c = aplicar_descuento_volumen({'producto': 'cable', 'precio': 20000, 'cantidad': 2})
assert _c['descuento_pct'] == 0.0
assert _c['precio_con_descuento'] == 20000
print("✅ Ejercicio 3: tests adicionales superados.")

## Ejercicio 4 · Calcular subtotales con map y lambda

Implementa `calcular_subtotales(pedidos_con_descuento)` que reciba la lista de pedidos ya enriquecidos (con `precio_con_descuento`) y devuelva una **nueva lista** con un campo adicional `'subtotal'` = `precio_con_descuento * cantidad`.

Usa `map` y `lambda`.

**Ejemplo:** `{'precio_con_descuento': 40500, 'cantidad': 5}` → añade `subtotal: 202500`.

In [ ]:
def calcular_subtotales(pedidos_con_descuento):
    return list(map(
        lambda p: {**p, 'subtotal': p['precio_con_descuento'] * p['cantidad']},
        pedidos_con_descuento
    ))

In [ ]:
# === Tests visibles · Ejercicio 4 ===
_pd = [{'producto': 'mouse', 'precio': 45000, 'cantidad': 5, 'precio_con_descuento': 40500, 'descuento_pct': 0.10}]
_st = calcular_subtotales(_pd)
assert _st[0]['subtotal'] == 202500
print("✅ Ejercicio 4: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 4 ===
_pd2 = [{'producto': 'a', 'precio': 100, 'cantidad': 3, 'precio_con_descuento': 100, 'descuento_pct': 0.0}]
assert calcular_subtotales(_pd2)[0]['subtotal'] == 300
assert calcular_subtotales([])==[]
print("✅ Ejercicio 4: tests adicionales superados.")

## Ejercicio 5 · Generar resumen de factura

Implementa `resumen_factura(pedidos_finales)` que reciba la lista de pedidos con todos los campos calculados y devuelva un diccionario:
- `'lineas'`: número de líneas de pedido
- `'subtotal_neto'`: suma de todos los subtotales
- `'iva'`: int, 19% del subtotal_neto
- `'total'`: int, subtotal_neto + iva
- `'ahorro_total'`: suma de lo que se ahorra en cada línea = `(precio_original - precio_con_descuento) * cantidad`

**Usa funciones anteriores** `parsear_pedidos`, `filtrar_nulos`, `aplicar_descuento_volumen` y `calcular_subtotales` en el pipeline.

No necesitas llamarlas aquí; solo usas la lista ya procesada.

In [ ]:
def resumen_factura(pedidos_finales):
    subtotal_neto = sum(p['subtotal'] for p in pedidos_finales)
    iva = int(subtotal_neto * 0.19)
    ahorro = sum(
        (p['precio'] - p['precio_con_descuento']) * p['cantidad']
        for p in pedidos_finales
    )
    return {
        'lineas':        len(pedidos_finales),
        'subtotal_neto': subtotal_neto,
        'iva':           iva,
        'total':         subtotal_neto + iva,
        'ahorro_total':  ahorro,
    }

In [ ]:
# === Tests visibles · Ejercicio 5 ===
_raw = ['laptop:2500000:2', 'mouse:45000:5', 'cargador:80000:0', 'teclado:120000:10']
_pedidos = parsear_pedidos(_raw)
_activos = filtrar_nulos(_pedidos)
_desc = list(map(aplicar_descuento_volumen, _activos))
_finales = calcular_subtotales(_desc)
_fac = resumen_factura(_finales)
assert _fac['lineas'] == 3
assert _fac['total'] == _fac['subtotal_neto'] + _fac['iva']
print("✅ Ejercicio 5: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 5 ===
assert _fac['ahorro_total'] >= 0
assert _fac['subtotal_neto'] > 0
_fac2 = resumen_factura([])
assert _fac2['lineas'] == 0
assert _fac2['total'] == 0
print("✅ Ejercicio 5: tests adicionales superados.")

## Ejercicio 6 · Imprimir factura formateada (pipeline completo)

Implementa `imprimir_factura(raw)` que reciba la lista de cadenas crudas (como en la Parte 1) y:
1. Llame a `parsear_pedidos` → `filtrar_nulos` → `map(aplicar_descuento_volumen)` → `calcular_subtotales` → `resumen_factura`.
2. Imprima cada línea de pedido con formato:
   `[producto] x[cantidad] @ $[precio_desc] = $[subtotal]  (descuento X%)`
3. Imprima el resumen al final:
   `Subtotal: $... | IVA: $... | TOTAL: $... | Ahorro: $...`
4. Devuelva el diccionario de `resumen_factura` (para los tests).

Esta función integra **todas** las piezas anteriores.

In [ ]:
def imprimir_factura(raw):
    pedidos  = parsear_pedidos(raw)
    activos  = filtrar_nulos(pedidos)
    desc     = list(map(aplicar_descuento_volumen, activos))
    finales  = calcular_subtotales(desc)
    resumen  = resumen_factura(finales)
    print('=' * 60)
    print('FACTURA')
    print('=' * 60)
    for p in finales:
        pct = int(p['descuento_pct'] * 100)
        linea = (p['producto'].ljust(12) +
                 ' x' + str(p['cantidad']) +
                 ' @ $' + '{:,}'.format(p['precio_con_descuento']) +
                 ' = $' + '{:,}'.format(p['subtotal']))
        if pct > 0:
            linea += '  (descuento ' + str(pct) + '%)'
        print(linea)
    print('-' * 60)
    print('Subtotal: $' + '{:,}'.format(resumen['subtotal_neto']) +
          ' | IVA: $' + '{:,}'.format(resumen['iva']) +
          ' | TOTAL: $' + '{:,}'.format(resumen['total']) +
          ' | Ahorro: $' + '{:,}'.format(resumen['ahorro_total']))
    return resumen

In [ ]:
# === Tests visibles · Ejercicio 6 ===
_raw = ['laptop:2500000:2', 'mouse:45000:5', 'cargador:80000:0', 'teclado:120000:10']
_resultado = imprimir_factura(_raw)
assert _resultado['lineas'] == 3
assert _resultado['total'] > 0
print("✅ Ejercicio 6: tests visibles superados.")

In [ ]:
# === Tests adicionales (ocultos) · Ejercicio 6 ===
assert _resultado['iva'] == int(_resultado['subtotal_neto'] * 0.19)
assert _resultado['ahorro_total'] >= 0
_r2 = imprimir_factura(['item:100:0'])
assert _r2['lineas'] == 0
print("✅ Ejercicio 6: tests adicionales superados.")

---
## ¡Proyecto terminado!

Si todos los tests pasan, construiste un procesador de pedidos funcional
que integra todos los conceptos de la clase:

| Parte | Concepto principal |
|---|---|
| 1 Parsear | bucle + split + conversión de tipos |
| 2 Filtrar nulos | `filter` + `lambda` |
| 3 Descuento por volumen | función con lógica de tramos + `{**dict}` |
| 4 Subtotales | `map` + `lambda` |
| 5 Resumen | `sum()` con comprehension, acumuladores |
| 6 Imprimir factura | **integración completa del pipeline** |

### Reto opcional (sin calificar)

- Añade soporte para `categoria` en cada pedido y calcula el total por
  categoría usando `agrupar_por` de la Práctica 02.
- Modifica `imprimir_factura` para que ordene las líneas por subtotal
  descendente antes de imprimir.
- ¿Qué pasaría si el precio unitario fuera 0? Añade validación.

> 💡 En la Clase 6, con pandas, este pipeline completo cabrá en menos de
> 10 líneas. Pero ahora entiendes cada transformación que pandas hace por
> dentro — y eso te convierte en un usuario que también puede depurar.